# TP 2 — Partie E : Fine-tuning : Mono-tâche vs Multi-tâche

Ce notebook implémente la **Partie E** du TP :
1. **Comparaison de deux PLM spécialisés sur les tweets arabes :**
   - `aubmindlab/bert-base-arabertv02-twitter`
   - `UBC-NLP/MARBERTv2` (spécifiquement entraîné sur 1 milliard de tweets dialectaux).
2. **E1. Modèles Mono-tâche (Sarcasme seul) :**
   - Fine-tuning end-to-end (110M de paramètres dégelés).
   - Perte pondérée ($w = 4{,}81$) pour gérer le déséquilibre, seuil optimisé sur `val`.
3. **E2. Modèles Multi-tâche (Sarcasme + Sentiment) :**
   - Encodeur partagé unique avec deux têtes de prédiction en parallèle (`Linear(h, 2)` pour sarcasme, `Linear(h, 3)` pour sentiment).
   - Perte combinée : $\mathcal{L} = \mathcal{L}_{\text{sarc}} + \lambda \cdot \mathcal{L}_{\text{sent}}$ avec $\lambda \in \{0{,}3 ; 0{,}5 ; 1{,}0\}$ évalué sur `val`.
4. **Protocole rigoureux imposé :**
   - Learning rate : `2e-5`, batch size : `32`, warmup : `10%`, weight decay : `0.01`, précision mixte `fp16`.
   - 3 graines (seeds 42, 43, 44) pour le modèle final (moyenne $\pm$ écart-type).
   - Journalisation dans `results/results.csv` via `ajouter_resultat`.

In [ ]:
# 1. Environnement et configuration
import os, sys
REPO = "/content/tp2-sarcasme-arabe"
if not os.path.exists(REPO):
    !git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!git pull

# Installation des bibliothèques nécessaires
!pip install -q transformers arabert pyarabic emoji accelerate

In [ ]:
# 2. Chargement des données et déduplication de train
import torch
import pandas as pd
import numpy as np
from src.seed import set_seed
from src.data import load_splits
from src.preprocess import dedupliquer_train
from src.results import ajouter_resultat, CHEMIN

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", device)

train_brut, val, test = load_splits(seed=42)
train = dedupliquer_train(train_brut, test, verbose=True)
print(f"Tailles : train={len(train)} | val={len(val)} | test={len(test)} (test scellé)")

# Poids de la classe sarcastique pour la perte pondérée
n_neg = (train["sarcasm"] == 0).sum()
n_pos = (train["sarcasm"] == 1).sum()
weights_sarc = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float32)
print(f"Poids appliqué à la classe sarcastique : {n_neg / n_pos:.2f}")

## 1. Partie E1 : Modèles Mono-Tâche (Sarcasme)
Fine-tuning complet de `aubmindlab/bert-base-arabertv02-twitter` et `UBC-NLP/MARBERTv2`.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from src.models_e import MonoTaskModel
from src.train_e import MultiTaskDataset, train_transformer

# -------------------------------------------------------------------------
# E1.1 : Fine-tuning Mono-tâche AraBERT-twitter
# -------------------------------------------------------------------------
MODEL_ARABERT = "aubmindlab/bert-base-arabertv02-twitter"
print(f"=== E1.1 : Fine-tuning Mono-tâche ({MODEL_ARABERT}) ===")
set_seed(42)
tok_arabert = AutoTokenizer.from_pretrained(MODEL_ARABERT)

ds_tr_ara = MultiTaskDataset(train["tweet"], train["sarcasm"].astype(int), None, tok_arabert, max_len=64)
ds_va_ara = MultiTaskDataset(val["tweet"], val["sarcasm"].astype(int), None, tok_arabert, max_len=64)
loader_tr_ara = DataLoader(ds_tr_ara, batch_size=32, shuffle=True)
loader_va_ara = DataLoader(ds_va_ara, batch_size=32, shuffle=False)

model_e1_arabert = MonoTaskModel(MODEL_ARABERT)
res_e1_arabert = train_transformer(
    model=model_e1_arabert,
    train_loader=loader_tr_ara,
    val_loader=loader_va_ara,
    device=device,
    weights_sarc=weights_sarc,
    is_multitask=False,
    lr=2e-5,
    epochs=4,
    patience=2,
)

ajouter_resultat({
    "modele": "E1. AraBERT fine-tuné mono-tâche",
    "preprocessing": "AutoTokenizer AraBERT",
    "graine": 42,
    "seuil": res_e1_arabert["seuil"],
    "pr_auc_val": res_e1_arabert["pr_auc_val"],
    "f1_sarc_val": res_e1_arabert["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_e1_arabert["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_e1_arabert["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_e1_arabert["temps_entrainement_s"],
    "params_entraines": res_e1_arabert["params_entraines"],
    "note": "seuil sur val, mono-tâche fine-tuné",
})

# Libération de mémoire GPU
del model_e1_arabert
torch.cuda.empty_cache()

In [ ]:
# -------------------------------------------------------------------------
# E1.2 : Fine-tuning Mono-tâche MARBERTv2
# -------------------------------------------------------------------------
MODEL_MARBERT = "UBC-NLP/MARBERTv2"
print(f"\n=== E1.2 : Fine-tuning Mono-tâche ({MODEL_MARBERT}) ===")
set_seed(42)
tok_marbert = AutoTokenizer.from_pretrained(MODEL_MARBERT)

ds_tr_mar = MultiTaskDataset(train["tweet"], train["sarcasm"].astype(int), None, tok_marbert, max_len=64)
ds_va_mar = MultiTaskDataset(val["tweet"], val["sarcasm"].astype(int), None, tok_marbert, max_len=64)
loader_tr_mar = DataLoader(ds_tr_mar, batch_size=32, shuffle=True)
loader_va_mar = DataLoader(ds_va_mar, batch_size=32, shuffle=False)

model_e1_marbert = MonoTaskModel(MODEL_MARBERT)
res_e1_marbert = train_transformer(
    model=model_e1_marbert,
    train_loader=loader_tr_mar,
    val_loader=loader_va_mar,
    device=device,
    weights_sarc=weights_sarc,
    is_multitask=False,
    lr=2e-5,
    epochs=4,
    patience=2,
)

ajouter_resultat({
    "modele": "E1. MARBERTv2 fine-tuné mono-tâche",
    "preprocessing": "AutoTokenizer MARBERTv2",
    "graine": 42,
    "seuil": res_e1_marbert["seuil"],
    "pr_auc_val": res_e1_marbert["pr_auc_val"],
    "f1_sarc_val": res_e1_marbert["f1_sarc_val"],
    "f1_sarc_val_seuil_0_5": res_e1_marbert["f1_sarc_val_seuil_0_5"],
    "accuracy_val_seuil": res_e1_marbert["accuracy_val_seuil"],
    "n_train": len(train),
    "n_val": len(val),
    "temps_entrainement_s": res_e1_marbert["temps_entrainement_s"],
    "params_entraines": res_e1_marbert["params_entraines"],
    "note": "seuil sur val, mono-tâche fine-tuné",
})

# Libération de mémoire GPU
del model_e1_marbert
torch.cuda.empty_cache()

## 2. Partie E2 : Apprentissage Multi-Tâche (Sarcasme + Sentiment)
Sur le meilleur modèle pré-entraîné (MARBERTv2), nous testons l'impact de l'apprentissage conjoint du sentiment avec $\lambda \in \{0{,}3 ; 0{,}5 ; 1{,}0\}$.

In [ ]:
from src.models_e import MultiTaskModel

# Dataset avec sentiment pour train et val
ds_tr_multi = MultiTaskDataset(train["tweet"], train["sarcasm"].astype(int), train["sentiment"], tok_marbert, max_len=64)
ds_va_multi = MultiTaskDataset(val["tweet"], val["sarcasm"].astype(int), val["sentiment"], tok_marbert, max_len=64)
loader_tr_multi = DataLoader(ds_tr_multi, batch_size=32, shuffle=True)
loader_va_multi = DataLoader(ds_va_multi, batch_size=32, shuffle=False)

resultats_multi = {}
for lam in [0.3, 0.5, 1.0]:
    print(f"\n=== E2 : Multi-tâche MARBERTv2 avec lambda = {lam} ===")
    set_seed(42)
    model_mt = MultiTaskModel(MODEL_MARBERT)
    res_mt = train_transformer(
        model=model_mt,
        train_loader=loader_tr_multi,
        val_loader=loader_va_multi,
        device=device,
        weights_sarc=weights_sarc,
        is_multitask=True,
        lam=lam,
        lr=2e-5,
        epochs=4,
        patience=2,
    )
    resultats_multi[lam] = res_mt
    ajouter_resultat({
        "modele": f"E2. MARBERTv2 multi-tâche (lambda={lam})",
        "preprocessing": "AutoTokenizer MARBERTv2",
        "graine": 42,
        "seuil": res_mt["seuil"],
        "pr_auc_val": res_mt["pr_auc_val"],
        "f1_sarc_val": res_mt["f1_sarc_val"],
        "f1_sarc_val_seuil_0_5": res_mt["f1_sarc_val_seuil_0_5"],
        "accuracy_val_seuil": res_mt["accuracy_val_seuil"],
        "n_train": len(train),
        "n_val": len(val),
        "temps_entrainement_s": res_mt["temps_entrainement_s"],
        "params_entraines": res_mt["params_entraines"],
        "note": f"seuil sur val, multi-tâche lambda={lam}",
    })
    del model_mt
    torch.cuda.empty_cache()

meilleur_lam = max(resultats_multi, key=lambda k: resultats_multi[k]["f1_sarc_val"])
print(f"\nMeilleur lambda retenu sur validation : {meilleur_lam} (F1: {resultats_multi[meilleur_lam]['f1_sarc_val']})")

## 3. Robustesse Statistique : 3 Graines Aléatoires pour le Modèle Final
Conformément au sujet, nous entraînons le meilleur modèle avec **3 graines aléatoires** (seeds 42, 43, 44) pour mesurer la moyenne et l'écart-type ($F1 \pm \sigma$).

In [ ]:
GRAINES = [42, 43, 44]
scores_final = []

print(f"=== Entraînement final multi-graines sur MARBERTv2 (lambda={meilleur_lam}) ===")
for seed in GRAINES:
    print(f"\n--> Exécution avec graine = {seed}")
    set_seed(seed)
    model_final = MultiTaskModel(MODEL_MARBERT)
    res_seed = train_transformer(
        model=model_final,
        train_loader=loader_tr_multi,
        val_loader=loader_va_multi,
        device=device,
        weights_sarc=weights_sarc,
        is_multitask=True,
        lam=meilleur_lam,
        lr=2e-5,
        epochs=4,
        patience=2,
    )
    scores_final.append(res_seed["f1_sarc_val"])
    ajouter_resultat({
        "modele": f"E2. Modèle Final MARBERTv2 MT (seed {seed})",
        "preprocessing": "AutoTokenizer MARBERTv2",
        "graine": seed,
        "seuil": res_seed["seuil"],
        "pr_auc_val": res_seed["pr_auc_val"],
        "f1_sarc_val": res_seed["f1_sarc_val"],
        "f1_sarc_val_seuil_0_5": res_seed["f1_sarc_val_seuil_0_5"],
        "accuracy_val_seuil": res_seed["accuracy_val_seuil"],
        "n_train": len(train),
        "n_val": len(val),
        "temps_entrainement_s": res_seed["temps_entrainement_s"],
        "params_entraines": res_seed["params_entraines"],
        "note": f"seed {seed}, modèle final retenu",
    })
    del model_final
    torch.cuda.empty_cache()

f1_mean = np.mean(scores_final)
f1_std = np.std(scores_final)
print(f"\nRésultat Final sur 3 graines : F1 sarc val = {f1_mean:.4f} ± {f1_std:.4f}")

## 4. Tableau de synthèse des résultats

In [ ]:
df_res = pd.read_csv(CHEMIN)
print(df_res[["modele", "graine", "seuil", "pr_auc_val", "f1_sarc_val", "f1_sarc_val_seuil_0_5", "temps_entrainement_s", "params_entraines"]].to_string())
